# 🔑 Solutions: Lesson 01 · The Artificial Neuron & the Perceptron

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 1.1: The planting neuron  &nbsp; `⭐`

Write `plant_decision(X, w, b)` that takes a batch `X` of shape `(n, 3)` and returns an integer array of 0/1 decisions
(1 if `z > 0`). Use **no loops**: one matrix–vector product.


In [2]:
def plant_decision(X, w, b):
    return (X @ w + b > 0).astype(int)

In [3]:
# ▶ self-check
X = np.array([[1,1,0],[1,0,0],[1,1,1],[0,1,0]])
out = plant_decision(X, np.array([3.,2.,-2.]), -3.)
dl.check("returns array of shape (4,)", out is not None and np.asarray(out).shape == (4,))
dl.check("decisions are [1, 0, 0, 0]", out is not None and list(out) == [1, 0, 0, 0],
         "(1,1,0): z=2>0 → plant; (1,0,0): z=0 → wait")

✅ returns array of shape (4,)
✅ decisions are [1, 0, 0, 0]


---
### 🧩 Problem 1.2: Design NAND and NOR  &nbsp; `⭐⭐`

**NAND** = NOT(AND): outputs 0 only when both inputs are 1. **NOR** = NOT(OR): outputs 1 only when both inputs are 0.
Choose weights and biases (fill the four variables) so the neurons reproduce the truth tables on all four inputs.
*(NAND is famous: every digital circuit can be built from NAND gates alone, so networks of neurons can compute anything a computer can!)*


In [4]:
w_nand, b_nand = np.array([-1., -1.]), 1.5
w_nor,  b_nor  = np.array([-1., -1.]), 0.5

In [5]:
# ▶ self-check
inp = np.array([[0,0],[0,1],[1,0],[1,1]])
ok = w_nand is not None and w_nor is not None
dl.check("NAND truth table", ok and list((inp @ np.asarray(w_nand) + b_nand > 0).astype(int)) == [1,1,1,0])
dl.check("NOR truth table",  ok and list((inp @ np.asarray(w_nor)  + b_nor  > 0).astype(int)) == [1,0,0,0])

✅ NAND truth table
✅ NOR truth table


**Why this works.** A negative weight turns "input is on" into evidence *against* firing. The bias sets how much total evidence is needed.

---
### 🧩 Problem 1.3: Majority vote of three  &nbsp; `⭐⭐`

Three sensors each say 1 ("disease") or 0 ("healthy"). Design a **single neuron** that outputs 1 when **at least two** of the three
sensors say 1. Fill in `w_maj` (length 3) and `b_maj`.


In [6]:
w_maj, b_maj = np.array([1., 1., 1.]), -1.5

In [7]:
# ▶ self-check
import itertools
combos = np.array(list(itertools.product([0, 1], repeat=3)))
want = (combos.sum(1) >= 2).astype(int)
got = (combos @ np.asarray(w_maj) + b_maj > 0).astype(int) if w_maj is not None else None
dl.check("majority gate correct on all 8 inputs", got is not None and np.array_equal(got, want))

✅ majority gate correct on all 8 inputs


---
### 🧩 Problem 1.4: Train your own perceptron  &nbsp; `⭐⭐`

Re-implement `train_perceptron(X, y, lr, epochs)` **without peeking** at the lesson. It must return `(w, b)`.
Stop early if an entire pass has zero mistakes. It should learn AND, OR and a linearly separable blob dataset perfectly.


In [8]:
def train_perceptron(X, y, lr=1.0, epochs=50):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            err = yi - int(np.dot(w, xi) + b > 0)
            if err:
                w, b, mistakes = w + lr * err * xi, b + lr * err, mistakes + 1
        if mistakes == 0:
            break
    return w, b

In [9]:
# ▶ self-check
from sklearn.datasets import make_blobs
inp = np.array([[0,0],[0,1],[1,0],[1,1]], float)
for name, t in [("AND", [0,0,0,1]), ("OR", [0,1,1,1])]:
    res = train_perceptron(inp, np.array(t))
    ok = res is not None and np.array_equal((inp @ res[0] + res[1] > 0).astype(int), t)
    dl.check(f"learns {name}", ok)
Xb, yb = make_blobs(60, centers=[[-2,-2],[2,2]], cluster_std=0.7, random_state=0)
res = train_perceptron(Xb, yb)
dl.check("100% accuracy on separable blobs", res is not None and ((Xb @ res[0] + res[1] > 0).astype(int) == yb).all())

✅ learns AND
✅ learns OR
✅ 100% accuracy on separable blobs


---
### 🧩 Problem 1.5: The boundary line (challenge)  &nbsp; `⭐⭐⭐`

For a 2-input neuron the decision boundary is the line $w_1x_1+w_2x_2+b=0$. Write `boundary_line(w, b)` returning
the **slope** and **intercept** of that line in the form $x_2 = \text{slope}\cdot x_1 + \text{intercept}$.
Then write `distance_to_boundary(x, w, b)` giving the **signed perpendicular distance** from point `x` to the line
(positive on the "fires" side). This distance is what *margin*-based methods like SVMs maximise.


In [10]:
def boundary_line(w, b):
    return -w[0] / w[1], -b / w[1]

def distance_to_boundary(x, w, b):
    return (np.dot(w, x) + b) / np.linalg.norm(w)

In [11]:
# ▶ self-check
w, b = np.array([2., 4.]), -4.
res = boundary_line(w, b)
dl.check("slope = -0.5, intercept = 1", res is not None and np.allclose(res, (-0.5, 1.0)))
d = distance_to_boundary(np.array([0., 0.]), w, b)
dl.check("origin is on the 'off' side, distance −4/√20", d is not None and np.isclose(d, -4 / np.sqrt(20)))
d0 = distance_to_boundary(np.array([0., 1.]), w, b)
dl.check("point on the line has distance 0", d0 is not None and np.isclose(d0, 0))

✅ slope = -0.5, intercept = 1
✅ origin is on the 'off' side, distance −4/√20
✅ point on the line has distance 0
